In [1]:
import re
import pandas as pd
import numpy as np
import nltk

In [7]:
script = pd.read_csv("C://Users//Nandhika//Desktop//LegallyBlonde//data//cleaned_df_of_script.csv")

In [3]:
script.head(5)

,line_id,speaker,type,text
0,1,NARRATOR,scene_heading,INT. DELTA GAMMA HOUSE - DAY ...
1,2,NARRATOR,scene_description,"flock of abstract, silky, golden strands -- PU..."
2,32,NARRATOR,scene_heading,INT. ELLE'S DELTA GAMMA ROOM - DAY - CONTINUOU...
3,33,NARRATOR,scene_description,"The CARD slides into the pink room, hitting th..."
4,41,NARRATOR,scene_description,"As he rises, we RISE WITH HIM, passing toned, ..."


In [8]:
script = script.drop(['line_id'],axis=1)
script.head(3)

,speaker,type,text
0,NARRATOR,scene_heading,INT. DELTA GAMMA HOUSE - DAY ...
1,NARRATOR,scene_description,"flock of abstract, silky, golden strands -- PU..."
2,NARRATOR,scene_heading,INT. ELLE'S DELTA GAMMA ROOM - DAY - CONTINUOU...


In [9]:
script.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3251 entries, 0 to 3250
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   speaker  3251 non-null   object
 1   type     3251 non-null   object
 2   text     3251 non-null   object
dtypes: object(3)
memory usage: 76.3+ KB


In [10]:
script['speaker'].value_counts()

speaker
ELLE                                                       1173
EMMETT                                                      192
WARNER                                                      162
DONOVAN                                                     158
PROFESSOR STROMWELL                                         123
                                                           ... 
AT THE DEFENSE TABLE                                          1
BROOKE                            .'..'                       1
REPORTER                                                      1
ELLE           ..                                     .       1
NERVOUS 1L GIRL                                               1
Name: count, Length: 150, dtype: int64

In [11]:
# Clean speaker column
script['speaker'] = (
    script['speaker']
    .astype(str)                      # ensure all are strings
    .str.strip()                      # remove leading/trailing spaces
    .str.replace(r'[^A-Za-z]', '', regex=True)  # remove special characters
    .str.upper()                      # convert to uppercase
)

In [12]:
script['speaker'].value_counts()

speaker
ELLE                 1285
EMMETT                199
WARNER                173
DONOVAN               165
SARAH                 127
                     ... 
NAME                    1
DICK                    1
ATTHEDEFENSETABLE       1
REPORTER                1
NERVOUSLGIRL            1
Name: count, Length: 84, dtype: int64

In [ ]:
filtered_df = script[script['speaker'].isin(["WARNER", "EMMETT"])]

In [16]:
filtered_df = filtered_df.reset_index(drop=True)

In [17]:
filtered_df.head(5)

,speaker,type,text
0,WARNER,dialogue,You're beautiful.
1,WARNER,stage_direction,(nervous)
2,WARNER,dialogue,You ready?
3,WARNER,dialogue,Her face is awash with devotion.
4,WARNER,dialogue,The reason I wanted to come here tonight


In [18]:
filtered_df.shape

(372, 3)

In [15]:
filtered_df['speaker'].value_counts()

speaker
EMMETT    199
WARNER    173
Name: count, dtype: int64

In [19]:
filtered_df= filtered_df.drop(['type'],axis=1)
filtered_df.head(3)

,speaker,text
0,WARNER,You're beautiful.
1,WARNER,(nervous)
2,WARNER,You ready?


### Classification of WARNER and EMMETT dialogues from Legally Blonde using BOW and TF-IDF

In [20]:
fd = filtered_df
fd.head(3)


,speaker,text
0,WARNER,You're beautiful.
1,WARNER,(nervous)
2,WARNER,You ready?


In [21]:
import nltk
import re
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Nandhika\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [22]:
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [24]:
# Cleaning
corpus = []
for i in range(0,len(fd)):
    review = re.sub('[^a-zA-Z]',' ',fd['text'][i])
    review = review.lower()
    review = review.split()
    review = [ps.stem(word) for word in review if word not in stopwords.words('english')]
    review = ''.join(review)
    corpus.append(review)

In [27]:
# Splitting the data into independent and dependent features
y = pd.get_dummies(fd['speaker'])
y = y.iloc[:,0].values

In [28]:
# Splitting into train test
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(corpus,y,test_size=0.2,random_state=43)

In [ ]:
# Bag of words
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=150, ngram_range=(3,2))
X_train = cv.fit_transform(X_train).toarray()
X_test = cv.transform(X_test).toarray()

In [30]:
# Model Training
from sklearn.naive_bayes import MultinomialNB
speaker_model = MultinomialNB().fit(X_train,y_train)

In [31]:
y_pred = speaker_model.predict(X_test)

In [32]:
## Model Evaluation
from sklearn.metrics import classification_report, accuracy_score
accuracy_score(y_test,y_pred)

0.5333333333333333

In [33]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

       False       0.00      0.00      0.00        33
        True       0.55      0.95      0.70        42

    accuracy                           0.53        75
   macro avg       0.27      0.48      0.35        75
weighted avg       0.31      0.53      0.39        75

